In [48]:
#import and initial hyperparameter fixing
import torch
import torch.nn as nn
import torch.nn.functional as F

device="cuda" if torch.cuda.is_available() else "cpu"

print("running on :",device)

#hyperparameters
batch_size=4 #number of batches model will see to observe
block_size=8 #number of letters to process(context length)
learning_rate=3e-4
max_iters=1000 #number of times the loop of backpropogation will run
eval_iters=200 #the loss will be the average off this many batches


running on : cpu


In [49]:
#cuda check
import torch

print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.12.1+cpu
CUDA version: None
CUDA available: False
GPU count: 0


In [50]:
import sys

print(sys.executable)

c:\Users\Arijeet\AppData\Local\Programs\Python\Python312\python.exe


In [51]:
#batch loading and vocabulary
with open("wizard_of_oz.txt","r",encoding="utf-8") as f:
    text=f.read()

print(text[:500])

print(len(text))

chars=sorted(list(set(text)))
print(chars)

vocab_size=len(chars)
print(vocab_size)

  
To My Readers


It's no use; no use at all. The children won't let me stop telling tales
of the Land of Oz. I know lots of other stories, and I hope to tell
them, some time or another; but just now my loving tyrants won't allow
me. They cry: "Oz--Oz! more about Oz, Mr. Baum!" and what can I do but
obey their commands?

This is Our Book--mine and the children's. For they have flooded me with
thousands of suggestions in regard to it, and I have honestly tried to
adopt as many of these suggesti
231881
['\n', ' ', '!', '"', '&', "'", '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\ufeff']
80


In [52]:
#conversion

stoi= {ch:i for i,ch in enumerate(chars)}
print(stoi)
itos={i:ch for i,ch in enumerate(chars)}
print(itos)

{'\n': 0, ' ': 1, '!': 2, '"': 3, '&': 4, "'": 5, '(': 6, ')': 7, ',': 8, '-': 9, '.': 10, '0': 11, '1': 12, '2': 13, '3': 14, '4': 15, '5': 16, '6': 17, '7': 18, '8': 19, '9': 20, ':': 21, ';': 22, '?': 23, 'A': 24, 'B': 25, 'C': 26, 'D': 27, 'E': 28, 'F': 29, 'G': 30, 'H': 31, 'I': 32, 'J': 33, 'K': 34, 'L': 35, 'M': 36, 'N': 37, 'O': 38, 'P': 39, 'Q': 40, 'R': 41, 'S': 42, 'T': 43, 'U': 44, 'V': 45, 'W': 46, 'X': 47, 'Y': 48, 'Z': 49, '[': 50, ']': 51, '_': 52, 'a': 53, 'b': 54, 'c': 55, 'd': 56, 'e': 57, 'f': 58, 'g': 59, 'h': 60, 'i': 61, 'j': 62, 'k': 63, 'l': 64, 'm': 65, 'n': 66, 'o': 67, 'p': 68, 'q': 69, 'r': 70, 's': 71, 't': 72, 'u': 73, 'v': 74, 'w': 75, 'x': 76, 'y': 77, 'z': 78, '\ufeff': 79}
{0: '\n', 1: ' ', 2: '!', 3: '"', 4: '&', 5: "'", 6: '(', 7: ')', 8: ',', 9: '-', 10: '.', 11: '0', 12: '1', 13: '2', 14: '3', 15: '4', 16: '5', 17: '6', 18: '7', 19: '8', 20: '9', 21: ':', 22: ';', 23: '?', 24: 'A', 25: 'B', 26: 'C', 27: 'D', 28: 'E', 29: 'F', 30: 'G', 31: 'H', 32:

In [53]:
#encoding annd deconding
def encode(text):
    return [stoi[c] for c in text]

encode("cat")

def decode(num):
    return "".join([itos[i] for i in num])

decode([55,53,72])

'cat'

In [54]:
sample="dorothy"
encoded=encode(sample)
decoded=decode(encoded)

print(sample)
print(encoded)
print(decoded)

dorothy
[56, 67, 70, 67, 72, 60, 77]
dorothy


In [55]:
#tensor
data = torch.tensor(encode(text), dtype=torch.long)

print(data[:30])

tensor([79,  1,  1,  0, 43, 67,  1, 36, 77,  1, 41, 57, 53, 56, 57, 70, 71,  0,
         0,  0, 32, 72,  5, 71,  1, 66, 67,  1, 73, 71])


In [56]:
#split
n=int(0.8* len(data))
train_data=data[:n]
val_data=data[n:]

In [57]:
#batch
def get_batch(split):
    data=train_data if split == "train" else val_data

    ix=torch.randint(len(data)-block_size,(batch_size,))
    x=torch.stack([data[i:i+block_size] for i in ix])

    y=torch.stack([data[i+1:i+1+block_size] for i in ix])

    x=x.to(device)
    y=y.to(device)

    return x ,y

In [58]:
x,y =get_batch("train")

print(x)
print(y)

tensor([[63, 71,  1, 75, 61, 72, 60, 61],
        [10,  1, 43, 60, 57,  1, 46, 61],
        [64, 61, 63, 57,  1, 53,  1, 60],
        [53, 66, 56,  1, 72, 60, 57,  1]])
tensor([[71,  1, 75, 61, 72, 60, 61, 66],
        [ 1, 43, 60, 57,  1, 46, 61, 78],
        [61, 63, 57,  1, 53,  1, 60, 73],
        [66, 56,  1, 72, 60, 57,  1, 68]])


In [59]:
class BigramLanguageModel(nn.Module):

    def __init__(self,vocab_size):
        super().__init__()

        self.token_embedding_table=nn.Embedding(vocab_size,vocab_size)

    def forward(self,index,targets=None):

        logits=self.token_embedding_table(index)

        if targets is None:
            loss=None

        else:
            B,T,C=logits.shape
            logits=logits.view(B*T,C)
            targets=targets.view(B*T)

            loss=F.cross_entropy(logits,targets)

        return logits,loss
    def generate(self, index, max_new_tokens):

        for _ in range(max_new_tokens):

            logits, loss = self.forward(index)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            index_next = torch.multinomial(probs, num_samples=1)

            index = torch.cat((index, index_next), dim=1)

        return index

In [60]:
model = BigramLanguageModel(vocab_size).to(device)

#training
optimizer=torch.optim.AdamW(model.parameters(),lr=learning_rate)

for iter in range(max_iters):
    xb,yb=get_batch("train")

    logits,loss=model(xb,yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if iter % 100 == 0:
        print(f"Iteration {iter}: Loss = {loss.item():.4f}")


Iteration 0: Loss = 5.0582
Iteration 100: Loss = 4.5856
Iteration 200: Loss = 4.6067
Iteration 300: Loss = 4.8296
Iteration 400: Loss = 4.7980
Iteration 500: Loss = 4.7883
Iteration 600: Loss = 4.8792
Iteration 700: Loss = 4.5848
Iteration 800: Loss = 4.7082
Iteration 900: Loss = 4.5810


In [61]:
context = torch.zeros((1,1), dtype=torch.long, device=device)

print(decode(model.generate(context, 500)[0].tolist()))


DiIF-F87ftgE:ww1C wZ&nr
&TR?_"IguP42O]IB8yNoJIv﻿("sM s.ek ﻿gPnJ?cT'2_DO]kh
s2X"(8ZvFGM0pzAF:gfa9g;!uPb1 vKzGN9,ve&PzJy(_s
woEVPzIF:,uZ.zJe﻿-Xpc.ytuGDj
zGQw!xSEyVTU72xLxX7cW,)uby:_aPR?1a:_DO)lCO]9dooIxxTPTTSo:PhIiID1!Hr_IiF1-K3 sEq;2x; tYWxXjIowJ7,naO?uZj[IQav0NQjH2afM wI),DmTq﻿!LBgzPhIQhI_&Mt!-2UF5tH,J4ZMNprs5Nz5CJ3r'-!﻿6oO]Pp?F-OBho?B]jhr.iQ]Cf''Nnx(jhOtJhtha:ZOhm__Chant.(059(QyN)saCO]M!CY5hqluW6Z:U'';RU0VT'ZiI)Dr?&(IkG4!'[pkcIZ﻿U0L?CG
"kGY5e0Z]tYUaW.X.R?LFtu2H_f3r(Ym(eE-TJ7x:M_9kPzmh.Q]﻿zWGRAX
